In [1]:
!pip install timm torch torchvision matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.6 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.6 MB 2.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.6 MB 3.0 MB/s eta 0:00:01
   ------------------------------------ --- 2.4/2.6 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 2.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import torch
import torch.nn as nn
import timm

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

from collections import Counter
import json
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.5.1+cu121
True
NVIDIA GeForce MX570 A


In [9]:
BATCH_SIZE = 8
EPOCHS = 15
LR = 0.0001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using:", DEVICE)

Using: cuda


In [10]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomRotation(8),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.08,0.08),
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.RandomPerspective(
        distortion_scale=0.1,
        p=0.3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
train_dataset = datasets.ImageFolder(
    "dataset/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "dataset/val",
    transform=val_transform
)

print("Classes:")
print(train_dataset.classes)

NUM_CLASSES = len(
    train_dataset.classes
)

print(
    "Number of Classes:",
    NUM_CLASSES
)

print(
    "Train Images:",
    len(train_dataset)
)

print(
    "Val Images:",
    len(val_dataset)
)

print("\nClass Mapping:")

for idx, cls in enumerate(
    train_dataset.classes
):
    print(
        f"{idx} -> {cls}"
    )

Classes:
['aadhaar', 'driving_license', 'pan', 'passport', 'voterId']
Number of Classes: 5
Train Images: 1300
Val Images: 327

Class Mapping:
0 -> aadhaar
1 -> driving_license
2 -> pan
3 -> passport
4 -> voterId


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [13]:
from collections import Counter

targets = train_dataset.targets

class_counts = Counter(targets)

weights = []

for i in range(NUM_CLASSES):
    weights.append(
        len(targets) / class_counts[i]
    )

weights = torch.tensor(
    weights,
    dtype=torch.float32
)

weights = weights / weights.sum() * NUM_CLASSES

weights = weights.to(DEVICE)

print("Class Weights:")
print(weights)

Class Weights:
tensor([0.6038, 0.8978, 1.3418, 0.9898, 1.1668], device='cuda:0')


In [14]:
weights_pretrained = (
    EfficientNet_B0_Weights.DEFAULT
)

model = efficientnet_b0(
    weights=weights_pretrained
)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)
print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\nitis/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:02<00:00, 10.0MB/s]

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=5, bias=True)
)


In [15]:
print(next(model.parameters()).device)

cuda:0


In [16]:
criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

In [17]:
print(train_dataset.classes)

print("Train:",
      len(train_dataset))

print("Val:",
      len(val_dataset))

['aadhaar', 'driving_license', 'pan', 'passport', 'voterId']
Train: 1300
Val: 327


In [18]:
best_acc = 0

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # Validation

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = (
        100 * correct / total
    )

    avg_loss = (
        running_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Loss: {avg_loss:.4f}"
        f" | Val Acc: {accuracy:.2f}%"
    )

    # Save best model

    if accuracy > best_acc:

        best_acc = accuracy

        torch.save(
            model.state_dict(),
            "best_document_classifier.pth"
        )

        print(
            f"New Best Model Saved! "
            f"({accuracy:.2f}%)"
        )

Epoch 1/15 | Loss: 0.8403 | Val Acc: 97.25%
New Best Model Saved! (97.25%)
Epoch 2/15 | Loss: 0.2020 | Val Acc: 98.17%
New Best Model Saved! (98.17%)
Epoch 3/15 | Loss: 0.0985 | Val Acc: 98.78%
New Best Model Saved! (98.78%)
Epoch 4/15 | Loss: 0.0707 | Val Acc: 99.08%
New Best Model Saved! (99.08%)
Epoch 5/15 | Loss: 0.0624 | Val Acc: 99.08%
Epoch 6/15 | Loss: 0.0341 | Val Acc: 99.39%
New Best Model Saved! (99.39%)
Epoch 7/15 | Loss: 0.0289 | Val Acc: 99.39%
Epoch 8/15 | Loss: 0.0232 | Val Acc: 98.47%
Epoch 9/15 | Loss: 0.0293 | Val Acc: 99.69%
New Best Model Saved! (99.69%)
Epoch 10/15 | Loss: 0.0237 | Val Acc: 99.69%
Epoch 11/15 | Loss: 0.0162 | Val Acc: 99.69%
Epoch 12/15 | Loss: 0.0149 | Val Acc: 99.08%
Epoch 13/15 | Loss: 0.0156 | Val Acc: 99.69%
Epoch 14/15 | Loss: 0.0199 | Val Acc: 99.69%
Epoch 15/15 | Loss: 0.0121 | Val Acc: 99.69%


In [19]:

with open(
    "classes.json",
    "w"
) as f:

    json.dump(
        train_dataset.classes,
        f
    )

print("Model Saved")
print("Classes Saved")

Model Saved
Classes Saved


In [20]:
print(train_dataset.classes)

['aadhaar', 'driving_license', 'pan', 'passport', 'voterId']
